This language model calculates probability of the next token based on the past and current tokens(causal system).
In this case we use the complete plays of Shakespear and work ar the character level, so the vocabulary is 65 symboles and training takes minutes 

In [ ]:
import os
import time 
import urllib.request
import torch
from torch import nn
import matplotlib.pyplot as plt
%matplotlib inline

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()

chars = sorted(set(text))
vocab_size = len(chars)

 #encoding chars to numbers
stoi= {c: i for i,c in enumerate(chars)}

#decoding numbers to chars
itos= {i: c for c,i in stoi.items()}

data = torch.tensor([stoi[c] for c in text], dtype = torch.long)
#90% of data is used for training and the rest is for the validation 
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

print("characters:", len(text), "| vocabulary:", vocab_size)
print("sample:", repr(text[:90]))

A token embedding table turns each chars id into a vector.
A position embedding adds the position, because attention only is order-blind.
then 4 transformer layers, also a causal mask is now switched on so no position can read the future(by setting the future vals equal to -infinity, when the vector is inserted as an input to the softMax the output would be zero)

In [ ]:
BLOCK, BATCH, D_MODEL, HEADS, LAYERS = 128, 64, 128, 4, 4

class TransformerLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.ReLU(), nn.Linear(4 * d_model, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        #norm1 is used before attention and norm2 is used before the feed-forward network
        h = self.norm1(x)
        x = x + self.attention(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.feed_forward(self.norm2(x))
#Generative pre-trained Transformers        
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, D_MODEL) #65 × 128(char ids with the vectors), the embedding vectors are learned during training
        self.position_embedding = nn.Embedding(BLOCK, D_MODEL)
        self.layers = nn.ModuleList([TransformerLayer(D_MODEL, HEADS) for _ in range(LAYERS)])
        self.norm = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, vocab_size)
#defines how data flows thriught those parts
    def forward(self, idx):
        T = idx.shape[1] #sequence length 
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=idx.device), diagonal=1)
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))
        for layer in self.layers:
            x = layer(x, mask)
        return self.head(self.norm(x))
        
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
model = TinyGPT().to(device)

print("device:", device)
print("parameters:", sum(p.numel() for p in model.parameters()))        

Batches and sampling 
a training example is a window of 128 chars, and its label is the same window shifted by 1, so every position predicts its own successor. Genrations samples one char at a time form the model's pitput distribution, divided a temperature that controls how sharp the distribution is(like if you increase the temp ofthe weather attoms can move more rapidly in comparison to the time that the weather is cold, so much of heat can cause hellucination)

In [ ]:
def get_batch(source):
    idx = torch.randint(len(source) - BLOCK - 1, (BATCH,))
    x = torch.stack([source[i:i + BLOCK] for i in idx])
    y = torch.stack([source[i + 1:i + BLOCK + 1] for i in idx])
    return x.to(device), y.to(device)


@torch.no_grad()
def generate(model, seed="ROMEO:", n=200, temperature=0.8):
    model.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in seed]], device=device)
    for _ in range(n):
        logits = model(idx[:, -BLOCK:])[:, -1, :] / temperature
        probabilities = torch.softmax(logits, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probabilities, 1)], dim=1)
    model.train()
    return "".join(itos[i] for i in idx[0].tolist())


x, y = get_batch(train_data)
print("input batch:", tuple(x.shape), "| target batch:", tuple(y.shape))
print("before training:", repr(generate(model, n=60)))

In [ ]:
STEPS = 2000 if device == "cuda" else 800          # keeps the runtime near a few minutes
CHECKPOINTS = [0, 200, STEPS // 2, STEPS]

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
history = []

start = time.time()
for step in range(STEPS + 1):
    if step in CHECKPOINTS:
        with torch.no_grad():
            vx, vy = get_batch(val_data)
            val_loss = loss_fn(model(vx).reshape(-1, vocab_size), vy.reshape(-1)).item()
        print(f"step {step:5d} | val loss {val_loss:.3f} | {time.time() - start:.0f}s")
        print(generate(model, n=110))
        print("-" * 70)

    x, y = get_batch(train_data)
    loss = loss_fn(model(x).reshape(-1, vocab_size), y.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append(loss.item())

print(f"trained in {time.time() - start:.0f}s on {device}")

In [ ]:
smoothed = torch.tensor(history).unfold(0, 50, 25).mean(dim=1)

plt.figure(figsize=(8, 3.5))
plt.plot(torch.arange(len(smoothed)) * 25, smoothed)
plt.xlabel("training step"); plt.ylabel("training loss")
plt.title("the tiny GPT learning to write")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for temperature in (0.2, 0.8, 1.5):
    print(f"=== temperature {temperature} ===")
    print(generate(model, n=140, temperature=temperature))
    print()

In [ ]:
prompt = "First Citizen:\nWe are all "
with torch.no_grad():
    logits = model(torch.tensor([[stoi[c] for c in prompt]], device=device))[0, -1].cpu()

top = logits.topk(10).indices
width = 0.27

plt.figure(figsize=(9, 3.5))
for offset, temperature in zip((-width, 0, width), (0.2, 0.8, 1.5)):
    probabilities = torch.softmax(logits / temperature, dim=-1)[top]
    plt.bar(torch.arange(10) + offset, probabilities, width, label=f"T = {temperature}")
plt.xticks(range(10), [repr(itos[int(i)]) for i in top])
plt.ylabel("probability"); plt.title('next character after "We are all "')
plt.legend(); plt.grid(axis="y", alpha=0.3)
plt.show()